In [28]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict,Annotated
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from pydantic import BaseModel,Field
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage,BaseMessage

load_dotenv()

True

In [29]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile"   # You can change model here
)

In [30]:
class JokeState(TypedDict):

    topic:str
    joke:str
    explanation:str

In [31]:
def generate_joke(state:JokeState):

    prompt = f'Generate a joke about {state["topic"]}'
    response = llm.invoke(prompt).content

    return {"joke": response}

In [32]:
def explain_joke(state:JokeState):

    prompt = f'Explain the joke: {state["joke"]}'
    response = llm.invoke(prompt).content

    return {"explanation": response}

In [33]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke',generate_joke)
graph.add_node('explain_joke',explain_joke)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'explain_joke')
graph.add_edge('explain_joke', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [34]:
config={"configurable": {"thread_id": "1"}}
workflow.invoke({"topic": "programming"}, config=config)

{'topic': 'programming',
 'joke': 'Why do programmers prefer dark mode?\n\nBecause light attracts bugs.',
 'explanation': 'A clever play on words. The joke is funny because it has a double meaning:\n\n1. In programming, a "bug" refers to an error or a flaw in the code.\n2. In everyday life, "bugs" can also refer to insects, which are attracted to light sources.\n\nSo, the joke is saying that programmers prefer dark mode (a display setting with a dark background and light text) because "light attracts bugs." This is a wordplay on the two meanings of "bugs." It\'s implying that if programmers use light mode (a bright background), they\'ll attract errors (programming bugs) to their code, but also making a humorous reference to the fact that insects are attracted to light.\n\nIt\'s a lighthearted and clever joke that requires a basic understanding of programming terminology and a bit of wordplay.'}

In [35]:
workflow.get_state(config=config)

StateSnapshot(values={'topic': 'programming', 'joke': 'Why do programmers prefer dark mode?\n\nBecause light attracts bugs.', 'explanation': 'A clever play on words. The joke is funny because it has a double meaning:\n\n1. In programming, a "bug" refers to an error or a flaw in the code.\n2. In everyday life, "bugs" can also refer to insects, which are attracted to light sources.\n\nSo, the joke is saying that programmers prefer dark mode (a display setting with a dark background and light text) because "light attracts bugs." This is a wordplay on the two meanings of "bugs." It\'s implying that if programmers use light mode (a bright background), they\'ll attract errors (programming bugs) to their code, but also making a humorous reference to the fact that insects are attracted to light.\n\nIt\'s a lighthearted and clever joke that requires a basic understanding of programming terminology and a bit of wordplay.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '',

In [37]:
config1={"configurable": {"thread_id": "2"}}
workflow.invoke({"topic": "pizza"}, config=config1)

{'topic': 'pizza',
 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.',
 'explanation': 'A classic play on words. The joke is funny because it uses a pun to create a wordplay. \n\nIn this joke, "crusty" has a double meaning:\n1. A pizza has a crust, which is the outer layer of the pizza.\n2. "Crusty" can also mean being irritable, grumpy, or in a bad mood.\n\nSo, the joke says the pizza is feeling "a little crusty," which is a clever way of saying it\'s in a bad mood, while also referencing the fact that a pizza has a crust. It\'s a lighthearted and cheesy (no pun intended) joke that uses wordplay to create humor.'}

In [41]:
workflow.get_state_history()

TypeError: Pregel.get_state_history() missing 1 required positional argument: 'config'